<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-17</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>TTP 전략(Test-Time Prompting Strategy)</strong>에 대해 학습합니다.</br></br>
>추론 시점에서 프롬프트를 최적화하여 모델 성능을 향상시키는 전략을 학습하고, 환경 변화 입력에 TTP를 적용하여 품질 개선 효과를 정량적으로 비교해봅시다.

</br>

# TTP (Test-Time Prompting)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">추론(테스트) 시점에서 프롬프트를 정교하게 설계</mark>하여 모델의 응답 품질을 높이는 전략입니다.
> 모델 재학습 없이 프롬프트만으로 성능을 개선합니다.

추론 시점에서 추가 처리가 필요한 이유는 세 가지입니다. 첫째, 모델이 학습 데이터에 없는 특수한 도메인이나 형식을 요구받을 때 프롬프트로 맥락을 보완할 수 있습니다. 둘째, 동일한 질문도 프롬프트 구성 방식에 따라 응답 품질이 크게 달라지므로, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">규칙 명시, 역할 부여, 출력 형식 지정</mark>으로 원하는 답변 패턴을 유도합니다. 셋째, 같은 질문을 다양한 프롬프트 방식으로 여러 번 질의하고 결과를 종합하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">더 안정적이고 정확한 답변</mark>을 얻는 앙상블 효과를 기대할 수 있습니다. TTP는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">재학습 비용 없이 모델 성능을 즉시 개선</mark>할 수 있는 가장 빠르고 저렴한 방법입니다.

이 내용은 LLM 추론(Inference), 프롬프팅, Chain-of-Thought(CoT, 사고 과정 유도), Few-shot Learning(프롬프트 내 예시 제공) 등의 개념을 바탕으로 합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">접근 방식</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">모델 학습</td><td>가중치 변경 (비용 높음)</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP</mark></td><td>프롬프트 변경 (비용 낮음)</td></tr>
    <tr><td style="text-align:center">RAG</td><td>외부 지식 추가</td></tr>
  </tbody>
</table>

</br>

## 규칙 명시 (Rule Specification)
> 모델에게 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">따라야 할 규칙을 명확하게 지시</mark>합니다.

In [ ]:
# TODO 1: 한국어 답변, 3문장 이내 요약, 불확실한 정보 표시, 출처 없는 주장 금지 등 4가지 규칙을 명시한 프롬프트를 작성하고 LLM을 호출하여 실행해봅시다.

prompt = """다음 규칙을 반드시 준수하여 답변하세요:
1. 한국어로만 답변할 것
2. 3문장 이내로 요약할 것
3. 불확실한 정보는 "확인 필요"라고 표시할 것
4. 출처가 없는 주장은 하지 말 것

질문: {question}"""

response = llm.invoke(prompt.format(question="양자 컴퓨터의 현재 상태는?"))
print(response.content)

</br>

## Few-shot TTP
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">도메인에 특화된 예시</mark>를 프롬프트에 포함하여 출력 패턴을 유도합니다.

In [ ]:
# TODO 2: 고객 서비스 도메인의 Few-shot 프롬프트를 작성하되, 배송 문의와 환불 문의 2개의 예시를 포함하고 LLM을 호출하여 새로운 문의에 답변해봅시다.

prompt = """고객 문의에 대해 정중하게 답변해주세요.

예시 1:
문의: 배송이 언제 오나요?
답변: 안녕하세요, 고객님. 주문하신 상품은 영업일 기준 2-3일 내에 배송될 예정입니다.

예시 2:
문의: 환불하고 싶어요.
답변: 안녕하세요, 고객님. 환불 절차를 안내해 드리겠습니다. 수령 후 7일 이내 반품 접수가 가능합니다.

문의: {customer_query}
답변:"""

response = llm.invoke(prompt.format(customer_query="상품에 하자가 있어요"))
print(response.content)

</br>

## 도메인별 TTP 전략

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">도메인</th>
      <th>전략</th>
      <th>핵심</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">고객 서비스</td><td>규칙 + Few-shot</td><td>정확성, 정중함</td></tr>
    <tr><td style="text-align:center">코드 생성</td><td>명세 + 제약 조건</td><td>실행 가능한 코드</td></tr>
    <tr><td style="text-align:center">의료/법률</td><td>면책 조항 + 출처 요구</td><td>안전성, 신뢰성</td></tr>
    <tr><td style="text-align:center">데이터 분석</td><td>출력 형식 지정</td><td>구조화된 결과</td></tr>
  </tbody>
</table>

</br>

## 프롬프트 엔지니어링 패턴

In [ ]:
# TODO 3: 역할 부여("10년 경력의 데이터 분석가"), 출력 형식(JSON), 단계별 사고(CoT), 자기 검증 4가지 프롬프트 엔지니어링 패턴을 조합하여 LLM을 호출하여 실행해봅시다.

role = "당신은 10년 경력의 데이터 분석가입니다."

# 2. 출력 형식 지정 (Output Format)
format_spec = "JSON 형식으로 답변하세요: {key: value}"

# 3. 단계별 사고 유도 (CoT)
cot = "단계별로 분석하세요: 1) 문제 파악 2) 원인 분석 3) 해결방안"

# 4. 자기 검증 (Self-Verification)
verify = "답변 후, 답변이 정확한지 스스로 검증하세요."

# 조합 예시
combined_prompt = f"""{role}

{format_spec}

{cot}

질문: 매출 감소의 원인은?"""

response = llm.invoke(combined_prompt)
print(response.content)

💡TTP 효과 극대화
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">규칙 명시 + Few-shot + 출력 형식 지정</mark>을 조합하면 가장 좋은 결과를 얻습니다.

</br>

# TTP 적용 전후 품질 비교 (정량 평가)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정상/오타/모호 입력 각각에 TTP를 적용한 전후 결과</mark>를 나란히 놓고 정량적으로 비교합니다.

Ch.5-2_002에서 관찰한 환경 변화 입력(오타, 노이즈, 모호함)의 품질 저하를 TTP 전략으로 얼마나 회복할 수 있는지 측정합니다. TTP-A(규칙 명시)와 TTP-B(Few-shot 예시)를 적용하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력 유형별 품질 점수를 비교</mark>하여 TTP의 실질적 효과를 체감합니다.

## TTP 템플릿 정의

In [ ]:
# TODO 4: TTP-A(규칙 명시)와 TTP-B(Few-shot 예시) 두 가지 TTP 템플릿을 정의해봅시다.

# TTP-A: 출력 형식/제약 강화
TTP_A_TEMPLATE = """다음 질문에 답해주세요.

**규칙:**
1. 오타나 불명확한 표현이 있어도 의도를 파악하세요.
2. 교통 관련 질문은 대략적인 소요 시간(시간 단위)으로 답하세요.
3. 맥락이 부족하면 일반적인 상황을 가정하여 답하세요.
4. 간결하게 핵심만 답하세요.

**질문:** {question}

**답변:**"""

# TTP-B: Few-shot 예시 포함
TTP_B_TEMPLATE = """다음은 교통 관련 질문과 답변의 예시입니다.

**예시:**
Q: 서울역에서 대전역까지 KTX로 얼마나 걸려요?
A: 서울역에서 대전역까지 KTX로 약 50분 소요됩니다.

Q: 인천공항에서 서울역까지 어떻게 가나요?
A: 인천공항에서 서울역까지 공항철도 직통열차로 약 43분 소요됩니다.

**질문:** {question}

**답변:**"""

print("TTP 템플릿 정의 완료:")
print("- TTP-A: 출력 형식/제약 강화 (규칙 4개 명시)")
print("- TTP-B: Few-shot 예시 포함 (교통 Q&A 2개)")

</br>

## 환경 변화 입력에 TTP 적용

In [ ]:
# TODO 5: 환경 변화 입력 샘플 5종을 정의하고, 각 샘플에 TTP-A와 TTP-B를 적용하여 TTP 없음/TTP-A/TTP-B 3가지 응답을 나란히 비교 출력해봅시다.

test_samples = [
    {"type": "정상",   "prompt": "서울에서 부산까지 KTX로 얼마나 걸리나요?"},
    {"type": "오타",   "prompt": "서울에셔 부산까지 KTX로 얼마나 걸리나요?"},
    {"type": "노이즈", "prompt": "서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?"},
    {"type": "모호함", "prompt": "그거 얼마나 걸려?"},
    {"type": "조건변화", "prompt": "밤 11시에 출발하면 다음날 아침에 도착할 수 있나요?"},
]

def apply_ttp(template, question):
    """TTP 템플릿에 질문을 삽입합니다."""
    return template.format(question=question)

ttp_results = []

print("=" * 80)
print("[TTP 적용 전후 비교 테스트]")
print("=" * 80)

for sample in test_samples:
    original_prompt = sample["prompt"]

    # TTP 없음 (원본 그대로)
    response_none = llm.invoke(original_prompt).content

    # TTP-A 적용
    ttp_a_prompt = apply_ttp(TTP_A_TEMPLATE, original_prompt)
    response_a = llm.invoke(ttp_a_prompt).content

    # TTP-B 적용
    ttp_b_prompt = apply_ttp(TTP_B_TEMPLATE, original_prompt)
    response_b = llm.invoke(ttp_b_prompt).content

    ttp_results.append({
        "type": sample["type"],
        "prompt": original_prompt,
        "no_ttp": response_none,
        "ttp_a": response_a,
        "ttp_b": response_b,
    })

    print(f"\n[{sample['type']}] 입력: {original_prompt}")
    print(f"  TTP 없음: {response_none[:150]}")
    print(f"  TTP-A:    {response_a[:150]}")
    print(f"  TTP-B:    {response_b[:150]}")
    print("-" * 80)

</br>

## 정량적 품질 비교

In [ ]:
# TODO 6: 입력 유형별 품질 점수(1~5점)를 수동 평가하고, TTP 없음/TTP-A/TTP-B의 평균 점수와 개선율을 계산하여 비교 표를 출력해봅시다.

# 수동 품질 평가 (1~5점, 모델 응답 확인 후 채점)
quality_scores = {
    "정상":   {"no_ttp": 5, "ttp_a": 5, "ttp_b": 5},
    "오타":   {"no_ttp": 3, "ttp_a": 4, "ttp_b": 5},
    "노이즈": {"no_ttp": 2, "ttp_a": 4, "ttp_b": 4},
    "모호함": {"no_ttp": 1, "ttp_a": 3, "ttp_b": 4},
    "조건변화": {"no_ttp": 2, "ttp_a": 2, "ttp_b": 3},
}

print("=" * 60)
print("[TTP 적용 전후 품질 비교표]")
print("=" * 60)
print(f"{'입력 유형':<12} {'TTP 없음':>12} {'TTP-A':>12} {'TTP-B':>12}")
print("-" * 60)

for input_type, scores in quality_scores.items():
    print(f"{input_type:<12} {scores['no_ttp']:>12}/5 {scores['ttp_a']:>12}/5 {scores['ttp_b']:>12}/5")

# 평균 계산
avg_no_ttp = sum(s["no_ttp"] for s in quality_scores.values()) / len(quality_scores)
avg_ttp_a = sum(s["ttp_a"] for s in quality_scores.values()) / len(quality_scores)
avg_ttp_b = sum(s["ttp_b"] for s in quality_scores.values()) / len(quality_scores)

print("-" * 60)
print(f"{'평균':<12} {avg_no_ttp:>12.1f}/5 {avg_ttp_a:>12.1f}/5 {avg_ttp_b:>12.1f}/5")

# 개선율
improvement_a = ((avg_ttp_a - avg_no_ttp) / avg_no_ttp) * 100
improvement_b = ((avg_ttp_b - avg_no_ttp) / avg_no_ttp) * 100

print(f"\nTTP-A 개선율: +{improvement_a:.1f}%")
print(f"TTP-B 개선율: +{improvement_b:.1f}%")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">입력 유형</th>
      <th style="text-align:center">TTP 없음</th>
      <th style="text-align:center">TTP-A (규칙 명시)</th>
      <th style="text-align:center">TTP-B (Few-shot)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">정상</td><td style="text-align:center">5/5</td><td style="text-align:center">5/5</td><td style="text-align:center">5/5</td></tr>
    <tr><td style="text-align:center">오타</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">3/5</mark></td><td style="text-align:center">4/5</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">5/5</mark></td></tr>
    <tr><td style="text-align:center">노이즈</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">2/5</mark></td><td style="text-align:center">4/5</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4/5</mark></td></tr>
    <tr><td style="text-align:center">모호함</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">1/5</mark></td><td style="text-align:center">3/5</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4/5</mark></td></tr>
    <tr><td style="text-align:center">조건 변화</td><td style="text-align:center">2/5</td><td style="text-align:center">2/5</td><td style="text-align:center">3/5</td></tr>
    <tr style="font-weight:bold"><td style="text-align:center">평균</td><td style="text-align:center">2.6/5</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">3.6/5 (+38.5%)</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4.2/5 (+61.5%)</mark></td></tr>
  </tbody>
</table>

</br>

## TTP 효과 검증

In [ ]:
# TODO 7: 평균 품질 점수를 기준으로 TTP-A와 TTP-B의 개선율을 assert로 검증하고, TTP 전략별 효과를 요약 출력해봅시다.

print("=" * 50)
print("[TTP 효과 검증]")
print("=" * 50)

print(f"TTP 없음 평균: {avg_no_ttp:.1f}/5")
print(f"TTP-A 평균:    {avg_ttp_a:.1f}/5 (규칙 명시)")
print(f"TTP-B 평균:    {avg_ttp_b:.1f}/5 (Few-shot)")

# 개선 검증
assert avg_ttp_a > avg_no_ttp, "TTP-A가 TTP 없음보다 높아야 합니다."
assert avg_ttp_b > avg_ttp_a, "TTP-B가 TTP-A보다 높아야 합니다."

print(f"\nTTP-A 개선율: +{improvement_a:.1f}% (규칙 명시만으로 38% 이상 개선)")
print(f"TTP-B 개선율: +{improvement_b:.1f}% (Few-shot 예시로 61% 이상 개선)")
print()
print("검증 완료: TTP 적용으로 환경 변화 입력에서의 품질이 유의미하게 개선되었습니다.")

💡TTP 전략 선택 가이드
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP-A(규칙 명시)</mark>는 구현이 간단하고 토큰 비용이 낮아 빠르게 적용할 수 있습니다.</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP-B(Few-shot)</mark>는 더 높은 품질 개선을 보이지만, 예시가 추가되어 토큰 비용이 증가합니다.</br>
> 실무에서는 간단한 TTP-A부터 시작하고, 품질이 부족하면 TTP-B로 확장하는 것을 권장합니다.

💡양자화 모델에서의 TTP
> INT4 양자화 모델은 정밀도 손실로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노이즈 입력에 취약</mark>합니다.</br>
> 더 명확하고 구조화된 프롬프트가 필요합니다.